# 🍛 Food Vision — Step 1: Baseline Indian Food Classifier

**Project:** Food Vision (Computer Vision + MLOps)
**Author:** Ankit Parate
**Notebook goal:** Fine-tune a pretrained **EfficientNet-B0** to classify Indian dishes from photos, and export everything the API (Step 4) will need.

### What this notebook does
| # | Section | Output |
|---|---------|--------|
| 1 | Config & setup | Reproducible seed, device check |
| 2 | Locate & validate data | Clean list of image paths (corrupt files removed) |
| 3 | EDA | Class balance chart, sample image grid |
| 4 | Stratified split | `splits/train.csv`, `val.csv`, `test.csv` |
| 5 | Transforms & loaders | Augmented train pipeline, deterministic eval pipeline |
| 6 | Model | EfficientNet-B0 (ImageNet weights) with a new classifier head |
| 7 | Two-phase training | Phase 1 head only → Phase 2 full fine-tune |
| 8 | Evaluation | Accuracy, Top-3, Macro-F1, confusion matrix, most-confused pairs |
| 9 | Export | `model.pt`, `class_names.json`, `metrics.json`, `history.csv` |
| 10 | Inference | `predict(image_path)` → top-3 dishes with confidence |

### How to run (Kaggle)
1. **Add Data** → search *"Indian Food Classification"* (by *l33tc0d3r*) → Add.
2. **Settings** → Accelerator: **GPU T4 x2** (one GPU is used), Internet: **On** (needed to download ImageNet weights).
3. **Save Version → Save & Run All (Commit)**. It runs in the background (~20–25 min) and keeps all outputs.
4. Open the finished version → **Output** tab → download `artifacts/`.

> Results of this run: **93.5% top-1, 97.5% top-3, 0.935 macro-F1** on 941 test images.

## 1. Configuration & Setup
All tunable values live in one `CONFIG` dict. In Step 2 these get logged to **MLflow**, so every experiment is traceable.

> The `FOOD_*` environment variables are only used for a quick local smoke test. On Kaggle you can ignore them.

In [ ]:
import os, json, random, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageFile

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score

warnings.filterwarnings("ignore")
ImageFile.LOAD_TRUNCATED_IMAGES = True  # tolerate slightly truncated JPEGs

CONFIG = {
    "data_dir": os.environ.get("FOOD_DATA_DIR", "/kaggle/input"),
    "output_dir": os.environ.get("FOOD_OUTPUT_DIR", "/kaggle/working/artifacts"),
    "seed": 42,
    "img_size": 224,
    "batch_size": 32,
    "num_workers": int(os.environ.get("FOOD_WORKERS", 2)),
    "val_size": 0.15,
    "test_size": 0.15,
    "model_name": "efficientnet_b0",
    "pretrained": os.environ.get("FOOD_PRETRAINED", "1") == "1",
    "head_epochs": int(os.environ.get("FOOD_HEAD_EPOCHS", 3)),      # phase 1: train classifier head only
    "finetune_epochs": int(os.environ.get("FOOD_FT_EPOCHS", 7)),    # phase 2: unfreeze whole network
    "head_lr": 1e-3,
    "finetune_lr": 3e-4,
    "weight_decay": 1e-4,
    "label_smoothing": 0.1,
    "max_images_per_class": int(os.environ.get("FOOD_MAX_PER_CLASS", 0)),  # 0 = use all
}

OUT = Path(CONFIG["output_dir"]); OUT.mkdir(parents=True, exist_ok=True)
(OUT / "splits").mkdir(exist_ok=True)

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True

seed_everything(CONFIG["seed"])
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
print(f"PyTorch {torch.__version__} | device: {DEVICE}"
      + (f" ({torch.cuda.get_device_name(0)})" if DEVICE.type == "cuda" else ""))

## 2. Locate & Validate the Data
Kaggle datasets often have nested folders (e.g. `Food Classification/<class_name>/*.jpg`), and the exact path can change between dataset versions. Rather than hard-coding a path, this cell **searches for the folder whose subfolders contain images**, so the notebook doesn't break.

It also opens every image once and **drops corrupt or unreadable files**. A single bad JPEG can crash training at epoch 6, so checking up front is cheaper.

In [ ]:
IMG_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

def find_image_root(base: Path) -> Path:
    """Return the directory with the most class-subfolders that contain images."""
    best, best_n = None, 0
    for d in [base, *[p for p in base.rglob("*") if p.is_dir()]]:
        subdirs = [s for s in d.iterdir() if s.is_dir()]
        n = sum(any(f.suffix.lower() in IMG_EXTS for f in s.iterdir()) for s in subdirs)
        if n > best_n:
            best, best_n = d, n
    if best is None or best_n < 2:
        raise FileNotFoundError(f"No class folders with images found under {base}. Did you add the dataset?")
    return best

IMAGE_ROOT = find_image_root(Path(CONFIG["data_dir"]))
print("Image root:", IMAGE_ROOT)

def is_valid_image(path: Path) -> bool:
    try:
        with Image.open(path) as im:
            im.verify()
        return True
    except Exception:
        return False

from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm

candidates = []
for class_dir in sorted(p for p in IMAGE_ROOT.iterdir() if p.is_dir()):
    files = sorted(f for f in class_dir.iterdir() if f.suffix.lower() in IMG_EXTS)
    if CONFIG["max_images_per_class"]:
        files = files[: CONFIG["max_images_per_class"]]
    candidates += [(f, class_dir.name) for f in files]

# Check 32 images at a time (Kaggle reads dataset files from network storage)
with ThreadPoolExecutor(max_workers=32) as ex:
    ok = list(tqdm(ex.map(lambda c: is_valid_image(c[0]), candidates),
                   total=len(candidates), desc="Checking images"))

records = [{"path": str(f), "label": lbl} for (f, lbl), good in zip(candidates, ok) if good]
bad = [str(f) for (f, _), good in zip(candidates, ok) if not good]

df = pd.DataFrame(records)
CLASS_NAMES = sorted(df["label"].unique())
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASS_NAMES)}
df["target"] = df["label"].map(CLASS_TO_IDX)

print(f"Valid images: {len(df):,} | Corrupt/skipped: {len(bad)} | Classes: {len(CLASS_NAMES)}")

## 3. Exploratory Data Analysis
Two checks before training:
- **Class balance.** If some dishes have far fewer images, plain accuracy can hide poor performance on them. That's why **Macro-F1** is reported later.
- **Visual sanity check.** Are labels correct? Are there watermarks, collages, or non-food images?

In [ ]:
counts = df["label"].value_counts()
plt.figure(figsize=(12, max(4, len(counts) * 0.28)))
sns.barplot(x=counts.values, y=counts.index, color="#d97706")
plt.title("Images per class"); plt.xlabel("Number of images"); plt.ylabel("")
plt.tight_layout(); plt.show()

print(f"Min: {counts.min()} ({counts.idxmin()}) | Max: {counts.max()} ({counts.idxmax()}) "
      f"| Imbalance ratio: {counts.max() / counts.min():.2f}x")

In [ ]:
def show_samples(frame, n_classes=12, seed=0):
    rng = np.random.default_rng(seed)
    chosen = rng.choice(CLASS_NAMES, size=min(n_classes, len(CLASS_NAMES)), replace=False)
    cols = 4; rows = int(np.ceil(len(chosen) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3.2, rows * 3.2))
    for ax, cls in zip(np.ravel(axes), chosen):
        path = frame[frame["label"] == cls].sample(1, random_state=seed)["path"].iloc[0]
        ax.imshow(Image.open(path).convert("RGB")); ax.set_title(cls, fontsize=10)
    for ax in np.ravel(axes):
        ax.axis("off")
    plt.tight_layout(); plt.show()

show_samples(df)

In [ ]:
# Image size distribution, which justifies the 224x224 resize choice
sizes = [Image.open(p).size for p in df["path"].sample(min(300, len(df)), random_state=0)]
w, h = zip(*sizes)
print(f"Width  median={np.median(w):.0f}  range=[{min(w)}, {max(w)}]")
print(f"Height median={np.median(h):.0f}  range=[{min(h)}, {max(h)}]")

## 4. Stratified Train / Validation / Test Split (70 / 15 / 15)
- **Stratified** means every split keeps the same class proportions.
- **Validation** is used to choose the best epoch. **Test** is touched only once, at the very end, so the reported score is honest.
- Splits are **saved to CSV**. This is the first MLOps habit: anyone, including future you, can reproduce exactly the same experiment.

In [ ]:
train_df, temp_df = train_test_split(
    df, test_size=CONFIG["val_size"] + CONFIG["test_size"],
    stratify=df["target"], random_state=CONFIG["seed"])
val_df, test_df = train_test_split(
    temp_df, test_size=CONFIG["test_size"] / (CONFIG["val_size"] + CONFIG["test_size"]),
    stratify=temp_df["target"], random_state=CONFIG["seed"])

for name, part in [("train", train_df), ("val", val_df), ("test", test_df)]:
    part.reset_index(drop=True).to_csv(OUT / "splits" / f"{name}.csv", index=False)
print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")

## 5. Transforms & Data Loaders
**Train augmentation** simulates real phone photos: different crops, angles, and lighting.
**Eval transform** is deterministic (resize → center crop), and it's **exactly what the API will use** in Step 4. A mismatch here is a classic production bug.

Normalization uses **ImageNet mean/std** because the pretrained weights expect it.

In [ ]:
IMAGENET_MEAN, IMAGENET_STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
S = CONFIG["img_size"]

train_tfms = transforms.Compose([
    transforms.RandomResizedCrop(S, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.25, hue=0.03),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_tfms = transforms.Compose([
    transforms.Resize(int(S * 1.14)),   # 256 for 224
    transforms.CenterCrop(S),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class FoodDataset(Dataset):
    def __init__(self, frame, tfms):
        self.paths = frame["path"].tolist(); self.targets = frame["target"].tolist(); self.tfms = tfms
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert("RGB")   # handles PNG alpha / grayscale
        return self.tfms(img), self.targets[i]

loader_kw = dict(batch_size=CONFIG["batch_size"], num_workers=CONFIG["num_workers"],
                 pin_memory=DEVICE.type == "cuda")
train_loader = DataLoader(FoodDataset(train_df, train_tfms), shuffle=True, drop_last=True, **loader_kw)
val_loader   = DataLoader(FoodDataset(val_df, eval_tfms), shuffle=False, **loader_kw)
test_loader  = DataLoader(FoodDataset(test_df, eval_tfms), shuffle=False, **loader_kw)
print(f"Batches → train {len(train_loader)} | val {len(val_loader)} | test {len(test_loader)}")

In [ ]:
# Visualize a few augmented training images (de-normalized)
xb, yb = next(iter(train_loader))
mean, std = torch.tensor(IMAGENET_MEAN).view(3, 1, 1), torch.tensor(IMAGENET_STD).view(3, 1, 1)
fig, axes = plt.subplots(1, 6, figsize=(16, 3))
for ax, x, y in zip(axes, xb[:6], yb[:6]):
    ax.imshow((x * std + mean).clamp(0, 1).permute(1, 2, 0)); ax.set_title(CLASS_NAMES[y], fontsize=9); ax.axis("off")
plt.suptitle("Augmented training samples"); plt.tight_layout(); plt.show()

## 6. Model: EfficientNet-B0 (Transfer Learning)
**Why EfficientNet-B0?**
- About 4M parameters (with a 20-class head) and strong ImageNet accuracy for its size, which makes it fast to train and cheap to serve on a CPU in production.
- It fits comfortably in 6GB of VRAM, so it also trains on a local RTX 3050.

The original 1000-class ImageNet head is replaced with a new `Dropout → Linear(num_classes)` head.

In [ ]:
def build_model(num_classes: int, pretrained: bool = True) -> nn.Module:
    weights = models.EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None
    m = models.efficientnet_b0(weights=weights)
    in_features = m.classifier[1].in_features
    m.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(in_features, num_classes))
    return m

model = build_model(len(CLASS_NAMES), CONFIG["pretrained"]).to(DEVICE)
total = sum(p.numel() for p in model.parameters())
print(f"Model: {CONFIG['model_name']} | params: {total/1e6:.2f}M | pretrained: {CONFIG['pretrained']}")

## 7. Two-Phase Training
| Phase | What's trainable | LR | Why |
|------|------------------|----|-----|
| 1 — Head warm-up | Classifier head only | 1e-3 | A random new head would send large, noisy gradients into the pretrained backbone and damage its features. |
| 2 — Full fine-tune | Entire network | 3e-4, cosine decay | The backbone adapts to food textures, like the difference between dal and sambar. |

Also used:
- **Label smoothing (0.1).** Many dishes look alike, so this stops the model from becoming overconfident.
- **AdamW** with weight decay, plus **mixed precision (AMP)** on GPU for roughly 2x speed.
- **Best-checkpoint saving** based on validation accuracy, not the last epoch.

In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=CONFIG["label_smoothing"])
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

def run_epoch(model, loader, optimizer=None):
    train = optimizer is not None
    model.train(train)
    total_loss, correct, n = 0.0, 0, 0
    with torch.set_grad_enabled(train):
        for x, y in loader:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            with torch.autocast(device_type=DEVICE.type, enabled=USE_AMP):
                logits = model(x); loss = criterion(logits, y)
            if train:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
            total_loss += loss.item() * x.size(0)
            correct += (logits.argmax(1) == y).sum().item(); n += x.size(0)
    return total_loss / n, correct / n

history, best_val_acc = [], 0.0
BEST_PATH = OUT / "model.pt"

def train_phase(phase, epochs, lr, trainable_params):
    global best_val_acc
    if epochs <= 0:
        return
    opt = torch.optim.AdamW(trainable_params, lr=lr, weight_decay=CONFIG["weight_decay"])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    for ep in range(1, epochs + 1):
        t0 = time.time()
        tr_loss, tr_acc = run_epoch(model, train_loader, opt)
        va_loss, va_acc = run_epoch(model, val_loader)
        sched.step()
        improved = va_acc > best_val_acc
        if improved:
            best_val_acc = va_acc; torch.save(model.state_dict(), BEST_PATH)
        history.append(dict(phase=phase, epoch=len(history) + 1, train_loss=tr_loss, train_acc=tr_acc,
                            val_loss=va_loss, val_acc=va_acc, lr=opt.param_groups[0]["lr"]))
        print(f"[{phase}] ep {ep}/{epochs} | train {tr_loss:.3f}/{tr_acc:.3f} | "
              f"val {va_loss:.3f}/{va_acc:.3f} | {time.time()-t0:.0f}s {'✓ saved' if improved else ''}")

# Phase 1: freeze backbone, train head
for p in model.features.parameters():
    p.requires_grad = False
train_phase("head", CONFIG["head_epochs"], CONFIG["head_lr"], model.classifier.parameters())

# Phase 2: unfreeze everything
for p in model.parameters():
    p.requires_grad = True
train_phase("finetune", CONFIG["finetune_epochs"], CONFIG["finetune_lr"], model.parameters())

hist_df = pd.DataFrame(history); hist_df.to_csv(OUT / "history.csv", index=False)
print(f"\nBest validation accuracy: {best_val_acc:.4f}")

## 8. Learning Curves
What to look for:
- **Train accuracy far above validation** points to overfitting. Fixes: more augmentation, more dropout, fewer epochs.
- **Both curves still rising** means the model is underfitting and more epochs would help.
- A jump at the phase boundary is expected, because unfreezing the backbone adds capacity.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, metric in zip(axes, ["loss", "acc"]):
    ax.plot(hist_df["epoch"], hist_df[f"train_{metric}"], "o-", label="train")
    ax.plot(hist_df["epoch"], hist_df[f"val_{metric}"], "o-", label="val")
    if CONFIG["head_epochs"] and CONFIG["finetune_epochs"]:
        ax.axvline(CONFIG["head_epochs"] + 0.5, ls="--", c="gray", lw=1)
        ax.text(CONFIG["head_epochs"] + 0.6, ax.get_ylim()[1], " unfreeze", va="top", fontsize=9, color="gray")
    ax.set_title(metric.capitalize()); ax.set_xlabel("Epoch"); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 9. Final Evaluation on the Held-Out Test Set
The **best checkpoint** is reloaded and evaluated once on the test set.

| Metric | Why it matters |
|--------|----------------|
| Top-1 accuracy | The headline number |
| **Top-3 accuracy** | The app shows 3 suggestions, so this is the metric users actually experience |
| **Macro-F1** | Treats every dish equally, regardless of class size |

In [ ]:
model.load_state_dict(torch.load(BEST_PATH, map_location=DEVICE))
model.eval()

all_probs, all_targets = [], []
with torch.no_grad():
    for x, y in test_loader:
        with torch.autocast(device_type=DEVICE.type, enabled=USE_AMP):
            logits = model(x.to(DEVICE))
        all_probs.append(torch.softmax(logits.float(), 1).cpu()); all_targets.append(y)
probs = torch.cat(all_probs).numpy(); y_true = torch.cat(all_targets).numpy()
y_pred = probs.argmax(1)

k = min(3, len(CLASS_NAMES))
top3 = np.argsort(-probs, axis=1)[:, :k]
metrics = {
    "test_top1_acc": float((y_pred == y_true).mean()),
    "test_top3_acc": float(np.mean([t in row for t, row in zip(y_true, top3)])),
    "test_macro_f1": float(f1_score(y_true, y_pred, average="macro")),
    "best_val_acc": float(best_val_acc),
    "num_classes": len(CLASS_NAMES),
    "num_test_images": int(len(y_true)),
}
for k_, v in metrics.items():
    print(f"{k_:>16}: {v:.4f}" if isinstance(v, float) else f"{k_:>16}: {v}")

print("\n" + classification_report(y_true, y_pred, labels=range(len(CLASS_NAMES)),
                                     target_names=CLASS_NAMES, digits=3, zero_division=0))

In [ ]:
cm = confusion_matrix(y_true, y_pred, labels=range(len(CLASS_NAMES)))
cm_norm = cm / cm.sum(1, keepdims=True).clip(min=1)
n = len(CLASS_NAMES)
plt.figure(figsize=(max(8, n * 0.45), max(6, n * 0.4)))
sns.heatmap(cm_norm, xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, cmap="Oranges",
            vmin=0, vmax=1, cbar_kws={"label": "Fraction of true class"})
plt.xlabel("Predicted"); plt.ylabel("True"); plt.title("Normalized confusion matrix (test set)")
plt.xticks(rotation=60, ha="right", fontsize=8); plt.yticks(fontsize=8)
plt.tight_layout(); plt.show()

### Most-confused dish pairs
This is the **most useful table for interviews**. It shows *where* the model fails and whether those failures make sense. For example, two visually similar fried snacks being confused is understandable; a sweet and a curry being confused is not.

In [ ]:
pairs = [(CLASS_NAMES[i], CLASS_NAMES[j], int(cm[i, j]), cm_norm[i, j])
         for i in range(n) for j in range(n) if i != j and cm[i, j] > 0]
confused = (pd.DataFrame(pairs, columns=["true", "predicted", "count", "rate"])
              .sort_values("count", ascending=False).head(10).reset_index(drop=True))
confused["rate"] = (confused["rate"] * 100).round(1).astype(str) + "%"
confused

In [ ]:
# Look at actual mistakes: often reveals label noise or genuinely ambiguous photos
wrong = np.where(y_pred != y_true)[0]
if len(wrong):
    pick = np.random.default_rng(0).choice(wrong, size=min(8, len(wrong)), replace=False)
    fig, axes = plt.subplots(2, 4, figsize=(14, 7))
    for ax in np.ravel(axes):
        ax.axis("off")
    for ax, i in zip(np.ravel(axes), pick):
        ax.imshow(Image.open(test_df["path"].iloc[i]).convert("RGB"))
        ax.set_title(f"true: {CLASS_NAMES[y_true[i]]}\npred: {CLASS_NAMES[y_pred[i]]} ({probs[i].max():.0%})",
                     fontsize=9, color="crimson")
    plt.suptitle("Misclassified test images"); plt.tight_layout(); plt.show()
else:
    print("No misclassifications on the test set 🎉")

## 10. Export Artifacts for the API
Everything the FastAPI service (Step 4) needs to load the model **without this notebook**:

| File | Purpose |
|------|---------|
| `model.pt` | Best weights (`state_dict`) |
| `class_names.json` | Index → dish name mapping (order matters!) |
| `model_config.json` | Architecture, image size, normalization, so the API preprocesses exactly like training |
| `metrics.json` | Test scores for the README and MLflow |
| `history.csv`, `splits/` | Reproducibility |

In [ ]:
with open(OUT / "class_names.json", "w") as f:
    json.dump(CLASS_NAMES, f, indent=2)
with open(OUT / "model_config.json", "w") as f:
    json.dump({"model_name": CONFIG["model_name"], "num_classes": len(CLASS_NAMES),
               "img_size": CONFIG["img_size"], "resize": int(CONFIG["img_size"] * 1.14),
               "mean": IMAGENET_MEAN, "std": IMAGENET_STD}, f, indent=2)
with open(OUT / "metrics.json", "w") as f:
    json.dump({**metrics, "config": CONFIG}, f, indent=2)

for p in sorted(OUT.rglob("*")):
    if p.is_file():
        print(f"{str(p.relative_to(OUT)):<24} {p.stat().st_size/1e6:7.2f} MB")

## 11. Inference Function (Prototype for the API)
This is the same logic the `/predict` endpoint will run: load an image → eval transform → softmax → top-3.

In [ ]:
@torch.no_grad()
def predict(image_path, top_k=3):
    img = Image.open(image_path).convert("RGB")
    x = eval_tfms(img).unsqueeze(0).to(DEVICE)
    p = torch.softmax(model(x).float(), 1)[0].cpu()
    conf, idx = p.topk(min(top_k, len(CLASS_NAMES)))
    return [{"dish": CLASS_NAMES[i], "confidence": round(float(c), 4)} for c, i in zip(conf, idx)]

sample = test_df.sample(1, random_state=7).iloc[0]
print("True label:", sample["label"])
for r in predict(sample["path"]):
    print(f"  {r['dish']:<20} {r['confidence']:.1%}")
plt.imshow(Image.open(sample["path"]).convert("RGB")); plt.axis("off"); plt.show()

## 12. Summary & Next Steps
**Done in this step:** a baseline classifier, honest test metrics, an error analysis, and portable artifacts.

**Write down your results** (from `metrics.json`) before Step 2, because they're the baseline every later experiment is compared against.

| Step | What's next |
|------|-------------|
| 2 | **MLflow experiment tracking.** Compare EfficientNet-B0 vs ConvNeXt-Tiny, with and without stronger augmentation |
| 3 | Nutrition lookup table (dish → approximate kcal / protein / carbs per serving) |
| 4 | **FastAPI** `/predict` service + upload UI, loading these exported artifacts |
| 5 | **Docker** + **GitHub Actions** CI (tests + image build) |
| 6 | Deploy to **Hugging Face Spaces** + README with results and demo GIF |